In [1]:
## Obligatory Black-Scholes PINN... 

import torch
import torch.autograd as tgrad

S = torch.Tensor([80]).requires_grad_() # stock price
t = torch.Tensor([0]).requires_grad_()  # time
sigma = torch.Tensor([0.3]).requires_grad_() # volatility \sigma
r = torch.Tensor([0.05]).requires_grad_() # risk free interest rate
K = torch.Tensor([70]) # strike  
T = torch.Tensor([1]) # option price
t2m = T-t # time to maturity

# intermediate black-scholes variables d1, d2
d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * t2m)/(sigma * torch.sqrt(t2m))
d2 = d1 - sigma * torch.sqrt(t2m)

#standard normal CDF
N0 = lambda value: 0.5 * (1 + torch.erf((value/2**0.5)))

# Nd1 is the option's Delta (how much the option price changes per $1 change in the stock price).
# Nd2 represents the risk-adjusted probability that the option will expire in-the-money.
Nd1 = N0(d1)
Nd2 = N0(d2)

# Recap: the greeks, V the derivative price
# \Delta: \partial V / \partial S, derivative price sensitivity to underlying proce
# \Theta: -\partial V / \partial t, the time sensitivity 
# \Gamma: \partial \Delta / \partial S, Delta sensitivity
# \Vega: \partial V / \prartial \sigma, the sensitivity to volatility
# \rho: \partial V / \partial r

# Black-Scholes formula
C = S* Nd1 - K* Nd2 *torch.exp(-r*t2m)

print("Option Price:", C) #17.01496

Option Price: tensor([17.0150], grad_fn=<SubBackward0>)


In [2]:
# computing the greeks

print(tgrad.grad(C, t, grad_outputs=torch.ones(C.shape), create_graph=True, only_inputs=True))

dCdt, dCdr, dCdvol, dCdS = tgrad.grad(C, (t, r, sigma, S), grad_outputs=torch.ones(C.shape), create_graph=True, only_inputs=True)
d2CdS2, = tgrad.grad(dCdS, S, grad_outputs=torch.ones(dCdS.shape), create_graph=True, only_inputs=True)

theta, delta, gamma, vega, rho = -dCdt, dCdS, d2CdS2, dCdvol, dCdr

greeks = dict(zip(["theta", "delta", "gamma", "vega", "rho"], [theta, delta, gamma, vega, rho]))

for k, v in greeks.items() :
    print(f'{k} : {v.item():.4f}')

# Theta 5.8385
# Delta 0.7769
# Gamma 0.0124
# Vega 23.8776
# Rho 45.1372

# The greeks satisfy the PDE
print((-theta + 0.5*sigma**2 * S**2*gamma + r*S*delta - r*C).item())
# 0

(tensor([-5.8385], grad_fn=<NegBackward0>),)
theta : 5.8385
delta : 0.7769
gamma : 0.0124
vega : 23.8776
rho : 45.1372
0.0


In [3]:
# PDE data

K = 40
r = 0.05
sigma = 0.25
T = 1
S_range = [0, 130]
t_range = [0, T]
payoff = lambda x: np.fmax(x-K, 0) # np.fmax is element by element comparison


def data_interior(n):
    """ sample t and S data on the interior of the domain """
    X = np.concatenate([np.random.uniform(*t_range, (n, 1)), 
                        np.random.uniform(*S_range, (n, 1))], axis=1) # sample t, S
    y = np.zeros((n, 1)) # residuals at these points is 0
    return X, y

def data_initial(n):
    """ """
    X = np.concatenate([np.ones((n, 1)),
                    np.random.uniform(*S_range, (n, 1))], axis=1) # stock price at time t = 1
    y = payoff(X[:, 1]).reshape(-1, 1) # reshape as a 2d matrix with 1 column, -1 tells numpy to calculate dims
    
    return X, y
  
def data_boundary(n):
    # generate data for times when stock proce hits 0, the payoff y1 is 0
    X1 = np.concatenate([np.random.uniform(*t_range, (n, 1)),
                        S_range[0] * np.ones((n, 1))], axis=1)
    y1 = np.zeros((n, 1)) 
    
    # generate data for times when stock proce hits 130, the payoff y2 is S - discounted value of strike
    # since option is gauranteed to be exercised. 
    X2 = np.concatenate([np.random.uniform(*t_range, (n, 1)), 
                        S_range[-1] * np.ones((n, 1))], axis=1)
    y2 = (S_range[-1] - K*np.exp(-r*(T-X2[:, 0].reshape(-1)))).reshape(-1, 1)
    
    return X1, y1, X2, y2


In [4]:
import torch.nn as nn
import torch.optim as optim
import numpy as np


class GalerkinNet(nn.Module):
  def __init__(self, input_dim, hidden_dim, n_layers=3, output_dim=1):
    super(GalerkinNet, self).__init__()
    self.input_dim = input_dim
    self.hidden_dim = hidden_dim
    self.output_dim = output_dim
    self.n = n_layers

    self.sig_act = nn.Tanh()

    self.Sw = nn.Linear(self.input_dim, self.hidden_dim)

    self.Uz = nn.Linear(self.input_dim, self.hidden_dim)
    self.Wsz = nn.Linear(self.hidden_dim, self.hidden_dim)

    self.Ug = nn.Linear(self.input_dim, self.hidden_dim)
    self.Wsg = nn.Linear(self.hidden_dim, self.hidden_dim)

    self.Ur = nn.Linear(self.input_dim, self.hidden_dim)
    self.Wsr = nn.Linear(self.hidden_dim, self.hidden_dim)
    
    self.Uh = nn.Linear(self.input_dim, self.hidden_dim)
    self.Wsh = nn.Linear(self.hidden_dim, self.hidden_dim)

    self.Wf = nn.Linear(hidden_dim, output_dim)
    

  def forward(self, x):
    S1 = self.Sw(x)
    for i in range(self.n):
      if i==0:
        S = S1
      else:
        S = self.sig_act(out)
      Z = self.sig_act(self.Uz(x) + self.Wsz(S))
      G = self.sig_act(self.Ug(x) + self.Wsg(S1))
      R = self.sig_act(self.Ur(x) + self.Wsr(S))
      H = self.sig_act(self.Uh(x) + self.Wsh(S*R))
      out = (1-G)*H + Z*S

    out = self.Wf(out)
    return out

model = GalerkinNet(2, 100, 3, 1)
model.cuda()

n_epochs = 60000
samples = {"pde": 5000, "bvp":5000, "ivp":5000}
mse_loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=3e-5)

In [5]:
def train(model, n_epochs, mse_loss, optimizer):
    loss_hist = []

    for epoch in range(n_epochs):
        
        # PDE Round
        X1, y1 = data_interior(samples['pde'])
        X1 = torch.from_numpy(X1).float().requires_grad_().cuda()
        y1 = torch.from_numpy(y1).float().cuda()
        
        # model predictions which is the option price
        y1_hat = model(X1)
        
        # take gradient with resect to [S, t] sampling points
        grads = tgrad.grad(y1_hat, X1, grad_outputs=torch.ones(y1_hat.shape).cuda(), retain_graph=True, create_graph=True, only_inputs=True)[0]
        # since model output is V, grads contains dVdt and dVdS
        dVdt, dVdS = grads[:, 0].view(-1, 1), grads[:, 1].view(-1, 1)
        
        # d2VdS2 is gamma, so need that
        grads2nd = tgrad.grad(dVdS, X1, grad_outputs=torch.ones(dVdS.shape).cuda(), create_graph=True, only_inputs=True)[0]
        d2VdS2 = grads2nd[:, 1].view(-1, 1)

        # pull out price data from [S, t] matrix    
        S1 = X1[:, 1].view(-1, 1)

        # get the interiro point pde mse_loss
        pde_loss = mse_loss(-dVdt, 0.5*((sigma*S1)**2)*d2VdS2 + r*S1*dVdS - r*y1_hat)
        
        
        # boundary error
        X21, y21, X22, y22 = data_boundary(samples['bvp'])
        
        X21 = torch.from_numpy(X21).float().cuda()
        y21 = torch.from_numpy(y21).float().cuda()
        
        X22 = torch.from_numpy(X22).float().cuda()
        y22 = torch.from_numpy(y22).float().cuda()
        
        y21_hat = model(X21)
        bvp1_loss = mse_loss(y21, y21_hat)
        
        y22_hat = model(X22)
        bvp2_loss = mse_loss(y22, y22_hat)
        
        
        # initial condition error
        X3, y3 = data_initial(samples['ivp'])
        
        X3 = torch.from_numpy(X3).float().cuda()
        y3 = torch.from_numpy(y3).float().cuda()
        
        y3_hat = model(X3)
        ivp_loss = mse_loss(y3, y3_hat)
        
        # Backpropagation and Update
        optimizer.zero_grad()
        combined_loss = pde_loss + bvp1_loss + bvp2_loss + ivp_loss
        combined_loss.backward()
        optimizer.step()
        
        loss_hist.append(combined_loss.item())
        if epoch % 500 == 0:
            print(f'{epoch}/{n_epochs} PDE Loss: {pde_loss.item():.5f}, BVP1 Loss: {bvp1_loss.item():.5f}, BVP2 Loss: {bvp2_loss.item():.5f}, IVP Loss: {ivp_loss.item():.5f},')

    return loss_hist

%matplotlib inline
import matplotlib.pyplot as plt
from pathlib import Path


retrain_model = False
model_path = "model_blackscholes_pinn.pth"
if not Path(model_path).exists() or retrain_model:
    loss_hist = train(model, n_epochs, mse_loss, optimizer)
    torch.save(model.state_dict(), model_path)
    print(f"model saved: {model_path}")
    plt.figure(figsize=(10, 6))
    plt.plot(loss_hist, color='blue', linewidth=2)
    plt.title('Training Loss')
    plt.ylabel('Total Training Loss')
    plt.xlabel('Epochs')

# Load the saved weights into it
loaded_model = GalerkinNet(2, 100, 3, 1)
loaded_model.load_state_dict(torch.load(model_path))


# Move it to GPU (if using CUDA) and set to evaluation mode
loaded_model.cuda() # send to GPU
loaded_model.eval() # switch to eval mode
print("model loaded")



model loaded


In [6]:


# test the model's accuracy today (t = 0) across various stock prices
test_time = 0.0
stock_prices = np.array([20, 30, 35, 40, 45, 50, 60, 80]) # Strike K is 40

# create pairs of [t, S] just like your training data did
X_test = np.zeros((len(stock_prices), 2))
X_test[:, 0] = test_time
X_test[:, 1] = stock_prices

X_test_tensor = torch.from_numpy(X_test).float().cuda()

# predict
with torch.no_grad(): # gradient not needed
    predictions = loaded_model(X_test_tensor)
    
pinn_prices = predictions.cpu().numpy().flatten()

# compare with the exact Black-Scholes formula
def exact_black_scholes(S, t, K=40, T=1, r=0.05, sigma=0.25):
    t2m = T - t
    if t2m <= 0: return np.maximum(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * t2m) / (sigma * np.sqrt(t2m))
    d2 = d1 - sigma * np.sqrt(t2m)
    N = lambda x: 0.5 * (1 + torch.erf(torch.tensor(x / 2**0.5)).item())
    
    # vectorize normal distribution helper for numpy array inputs
    Nd1 = np.array([N(x) for x in d1])
    Nd2 = np.array([N(x) for x in d2])
    return S * Nd1 - K * Nd2 * np.exp(-r * t2m)

# calculate exact prices
true_prices = exact_black_scholes(stock_prices, test_time)

# summarize
print(f"{'Stock Price':<15}{'Model Prediction':<25}{'True BS Price':<20}{'Absolute Error':<15}")
print("-" * 75)
for S, ai, true in zip(stock_prices, pinn_prices, true_prices):
    error = abs(ai - true)
    print(f"${S:<14.2f}${ai:<24.4f}${true:<19.4f}${error:<14.4f}")

Stock Price    Model Prediction         True BS Price       Absolute Error 
---------------------------------------------------------------------------
$20.00         $0.0060                  $0.0109             $0.0049        
$30.00         $0.7644                  $0.7685             $0.0042        
$35.00         $2.3107                  $2.3085             $0.0022        
$40.00         $4.9389                  $4.9344             $0.0045        
$45.00         $8.5101                  $8.4957             $0.0143        
$50.00         $12.7196                 $12.7063            $0.0134        
$60.00         $22.1214                 $22.1112            $0.0102        
$80.00         $41.9626                 $41.9566            $0.0060        
